In [ ]:
import pandas as pd
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

from utils.caption_standardizer import CaptionStandardizer
from utils.caption_tokenizer import CaptionTokenizer

df = pd.read_csv("../flickr8k/captions.txt")

images = df["image"].unique()

train_images, test_images = train_test_split(
    images,
    test_size=0.05,
    random_state=42,
    shuffle=True
)

train_df = df[df["image"].isin(train_images)]
test_df = df[df["image"].isin(test_images)]

print(f"Train images: {len(train_images)}")
print(f"Test images: {len(test_images)}")

print(f"Train captions: {len(train_df)}")
print(f"Test captions: {len(test_df)}")

In [ ]:
import json

from utils.vocabulary import Vocabulary
from utils.max_len import compute_max_len

train_captions: list[str] = train_df["caption"].tolist()
test_captions: list[str] = test_df["caption"].tolist()

standardizer = CaptionStandardizer()
train_captions = standardizer.standardize_all(train_captions)
test_captions = standardizer.standardize_all(test_captions)

tokenizer = CaptionTokenizer()
tok_train_captions = tokenizer.tokenize_all(train_captions)

vocabulary = Vocabulary()
vocabulary.build(tok_train_captions)

max_len = compute_max_len(tok_train_captions)

# + [START] ed [END]
max_len = max_len + 2

vectorizer = TextVectorization(
    max_tokens=None,
    standardize=None,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_len,
    pad_to_max_tokens=False,
    vocabulary=vocabulary.get_vocabulary(),
)

# Aggiunge '' (padding) e '[UNK]'
vocab = vectorizer.get_vocabulary()

config = {"vec_config": vectorizer.get_config(), "vocab_size": len(vocab), "max_len": max_len}

with open("../config/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"Vocab size: {vocabulary.size}")
print(f"Max caption len: {max_len}")

In [ ]:
# prima di salvare train caption e test caption (sono già standardizzate)
# dobbiamo troncarle a max_len token!

def truncate_to_max_len_tokens(caption: str, tokenizer: CaptionTokenizer, max_len: int):
    tokens = tokenizer.tokenize(caption)
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
    return " ".join(tokens)


train_captions = [truncate_to_max_len_tokens(caption, tokenizer, max_len) for caption in train_captions]
test_captions = [truncate_to_max_len_tokens(caption, tokenizer, max_len) for caption in test_captions]

print(f"Train captions: {len(train_captions)}")

train_df["caption"] = train_captions
test_df["caption"] = test_captions

train_df.to_csv("../flickr8k/captions_train.csv", index=False)
test_df.to_csv("../flickr8k/captions_test.csv", index=False)